In [3]:
pip install langgraph langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 10.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.2/563.2 kB 14.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.5 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.51.2━━━━━━━━━━━━━━━━━━━━  6/15 [openai]
    Uninstalling openai-1.51.2:━━━━━━━━━━━━━━━━━━━━━━━  6/15 [openai]
      Successfully uninstalled openai-1.51.2━━━━━━━━━━━━━━━━━━  6/15 [openai]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [langchain]15 [langgraph]openai]

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, ToolMessage, HumanMessage
from typing import Annotated
from typing_extensions import TypedDict
import json
import operator


In [6]:
class AgentState:(TypeDict):
    messages: Annotated[list, operator.add]

model = ChatOpenAI(model="gpt-4-0613", api_key="sk-xxx")

def get_weather(city: str) -> str:
    return f"{city} is sunny today."

def caculate(expression: str) -> str:
    try:
        return f"The result {eval(expression)}"
    except Exception as e:
        return f"Error: {e}"

tool_map = {
    "get_weather": get_weather,
    "caculate": caculate
}

tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "desctiption": "Get the weather of a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

model_with_tools = model.bind_tools(tools_schema)


def llm_node(state: AgentState):
    messages = state["messages"]
    response = model_with_tools.invoke(messages)
    return {"messages": [response]}

def tool_node(state: AgentState):
    last_message = state["messages"][-1]
    tool_result = []

    for  tool_call in last_message.tool_calls:
        name = tool_call["name"]
        args = tool_call["args"]

        result = tool_map[name](**args)

        tool_result.append(
            ToolMessage(
                content=result,
                tool_call_id=tool_call["id"]
            )
        )
    return {"messages": tool_result}

def should_continue(state: AgentState):
    last_message = state["messages"][-1]

    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "call_tool"
    else:
        return "end"

# create graph
graph_bulider = StateGraph(AgentState)
graph_bulider.add_node(llm_node, "llm")
graph_bulider.add_node(tool_node, "tool")

graph_bulider.set_entry_node("llm")

# create conditional edges
graph_bulider.add_continue_edge(
    "llm",
    should_continue,
    {
        "call_tool": "tool",
        "end": END
    }
)
# create fixed edge
graph_bulider.add_fixed_edge("tool", "llm")

agent = graph_bulider.compile()

result = agent.invoke({
    "messages": [
        SystemMessage(content="You are a helpful assistant."),
        HumanMessage(content="What is the weather in New York?")
    ]
})

print(result["messages"][-1].content)

SyntaxError: invalid syntax (686706563.py, line 1)